# GraphRAG — Hands-On

Offline construction of entity-relation graph, local retrieval, and community summaries.

## 0. Build triples with provenance

In [ ]:
%pip install -q numpy
from collections import defaultdict, deque
triples=[("GraphRAG","extends","RAG","doc1"),("GraphRAG","uses","knowledge graphs","doc2"),("knowledge graphs","represent","relations","doc3"),("RAG","uses","retrieval","doc4"),("retrieval","supports","grounding","doc5")]
g=defaultdict(list)
for s,r,o,p in triples: g[s].append((r,o,p))
print(dict(g))

## 1. Local neighborhood retrieval

In [ ]:
def neighborhood(seed, depth=1):
    seen={seed}; q=deque([(seed,0)]); facts=[]
    while q:
        node,d=q.popleft()
        for rel,dst,prov in g.get(node,[]):
            facts.append(f"{node} {rel} {dst} [{prov}]")
            if d < depth and dst not in seen: seen.add(dst); q.append((dst,d+1))
    return facts
print("\n".join(neighborhood("GraphRAG",2)))

## 2. Toy community summaries

In [ ]:
communities={"rag_methods":["GraphRAG","RAG","retrieval"],"graph_core":["knowledge graphs","relations"]}
summaries={name: "Community about " + ", ".join(nodes) for name,nodes in communities.items()}
print(summaries)

## 3. Query mode selection

In [ ]:
def mode(q): return "global" if any(w in q.lower() for w in ["overall","themes","summarize"]) else "local"
for q in ["How does GraphRAG use graphs?", "Summarize overall themes"]:
    print(q, "->", mode(q))

## 4. Answer assembly

In [ ]:
def retrieve_graph(q):
    if mode(q)=="global": return list(summaries.values())
    return neighborhood("GraphRAG", 1)
print("\n".join(retrieve_graph("How does GraphRAG use graphs?")))

## 5. Exercise prompts
1. Add entity aliases.
2. Add reverse edges.
3. Compare vector-style keyword search to graph neighborhood retrieval.